In [ ]:
import requests
import pandas as pd
import time
import os
import json

In [ ]:
# Download Data
# the results will be stored in the folder "data"
# the folder structure should be:
# project-chaggg/
# ├── data/
# │   └── chicago_crime_data/
# │       ├── batches/
# │       └── chicago_crimes_2001_2025.csv

# ── Config ────────────────────────────────────────────────────────────────────
APP_TOKEN    = "INSERT_YOUR_APP_TOKEN_HERE"  # Get your token from Chicago Data portal
BASE_URL     = "https://data.cityofchicago.org/resource/ijzp-q8t2.json"
BATCH_SIZE   = 50000
BASE_DIR = os.getcwd()
OUTPUT_DIR   = os.path.join(BASE_DIR, "data", "chicago_crime_data")
FINAL_OUTPUT = os.path.join(OUTPUT_DIR, "chicago_crimes_2001_2025.csv")
PROGRESS_FILE = os.path.join(OUTPUT_DIR, "progress.json")

BATCH_DIR = os.path.join(OUTPUT_DIR, "batches")
os.makedirs(BATCH_DIR, exist_ok=True)

EXPECTED_COLUMNS = [
    'id', 'case_number', 'date', 'block', 'iucr', 'primary_type',
    'description', 'location_description', 'arrest', 'domestic', 'beat',
    'district', 'ward', 'community_area', 'fbi_code', 'year', 'updated_on',
    'x_coordinate', 'y_coordinate', 'latitude', 'longitude', 'location'
]

# ── Progress helpers ──────────────────────────────────────────────────────────
def load_progress():
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE) as f:
            return json.load(f)
    return {"offset": 0, "batch_num": 1, "total_records": 0}

def save_progress(offset, batch_num, total):
    with open(PROGRESS_FILE, "w") as f:
        json.dump({"offset": offset, "batch_num": batch_num, "total_records": total}, f)

# ── Fetch ─────────────────────────────────────────────────────────────────────
def fetch_batch(offset, retries=3):
    params = {
        "$limit":      BATCH_SIZE,
        "$offset":     offset,
        "$order":      "date ASC",
        "$$app_token": APP_TOKEN
    }
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(BASE_URL, params=params, timeout=60)
            r.raise_for_status()
            return r.json()
        except requests.RequestException as e:
            print(f"  [Attempt {attempt}/{retries}] Failed: {e}")
            if attempt < retries:
                time.sleep(5 * attempt)
            else:
                raise

# ── Align columns ─────────────────────────────────────────────────────────────
def align_columns(df):
    for col in EXPECTED_COLUMNS:
        if col not in df.columns:
            df[col] = None
    return df[EXPECTED_COLUMNS]

# ── Download ──────────────────────────────────────────────────────────────────
def download_all():
    if os.path.exists(FINAL_OUTPUT):
        print(f"File already exists: {FINAL_OUTPUT}\nDownload skipped.")
        return
    progress  = load_progress()
    offset    = progress["offset"]
    batch_num = progress["batch_num"]
    total     = progress["total_records"]

    print(f"{'Resuming' if offset > 0 else 'Starting'} download "
          f"(batch={batch_num}, offset={offset}, saved={total})\n")

    while True:
        batch_file = os.path.join(BATCH_DIR, f"batch_{batch_num:04d}.csv")

        if os.path.exists(batch_file):
            print(f"Batch {batch_num} already exists, skipping...")
            offset    += BATCH_SIZE
            batch_num += 1
            continue

        print(f"Fetching batch {batch_num} (offset={offset})...")
        batch = fetch_batch(offset)

        if not batch:
            print("No more data. Download complete.")
            break

        df = align_columns(pd.DataFrame(batch))
        df.to_csv(batch_file, index=False)

        total     += len(batch)
        offset    += BATCH_SIZE
        batch_num += 1
        save_progress(offset, batch_num, total)
        print(f"  -> {len(batch)} records. Total: {total}")
        time.sleep(0.5)

    if os.path.exists(PROGRESS_FILE):
        os.remove(PROGRESS_FILE)

# ── Stream-merge batches ──────────────────────────────────────────────────────
def combine_batches():
    if os.path.exists(FINAL_OUTPUT):
        print(f"Final file already exists: {FINAL_OUTPUT}, skipping merge.")
        return

    batch_files = sorted(
        os.path.join(BATCH_DIR, f)
        for f in os.listdir(BATCH_DIR) if f.endswith(".csv")
    )
    print(f"\nMerging {len(batch_files)} batches into final CSV...")

    with open(FINAL_OUTPUT, "w", encoding="utf-8") as out:
        for i, path in enumerate(batch_files):
            df = pd.read_csv(path, low_memory=False)
            df = align_columns(df)                      # re-align just in case
            df.to_csv(out, index=False, header=(i == 0))
            print(f"  Merged {path} ({len(df)} rows)")

    print(f"\nDone! Saved to: {FINAL_OUTPUT}")

# ── Utility helpers ───────────────────────────────────────────────────────────
def convert_to_parquet(parquet_path=None):
    """Generate a parquet copy of the final CSV.
    Parquet is smaller and faster to load than CSV.
    If parquet already exists the function does nothing.
    """
    parquet_path = parquet_path or os.path.join(OUTPUT_DIR, "chicago_crimes_cleaned.parquet")
    if os.path.exists(parquet_path):
        print(f"Parquet already exists: {parquet_path}")
        return
    if not os.path.exists(FINAL_OUTPUT):
        raise FileNotFoundError("CSV not found. Run download_all() and combine_batches() first.")
    df = pd.read_csv(FINAL_OUTPUT, low_memory=False)
    df.to_parquet(parquet_path, index=False)
    print(f"Saved parquet: {parquet_path}")

def load_data(parquet_path=None, csv_path=None):
    """Load dataset, preferring parquet if available, otherwise falls back to CSV.

    Args:
        parquet_path: optional override for parquet location.
        csv_path: optional override for csv location.

    Returns:
        pandas.DataFrame containing the crime data.

    Raises:
        FileNotFoundError: if neither file is present.
    """
    parquet_path = parquet_path or os.path.join(OUTPUT_DIR, "chicago_crimes_cleaned.parquet")
    csv_path = csv_path or FINAL_OUTPUT

    if os.path.exists(parquet_path):
        print("Loading from parquet...")
        return pd.read_parquet(parquet_path)
    if os.path.exists(csv_path):
        print("Loading from CSV...")
        return pd.read_csv(csv_path, low_memory=False)
    raise FileNotFoundError(
        f"No data found. Run download_all() then combine_batches(), or place a file at {parquet_path} or {csv_path}.")

Starting download (batch=1, offset=0, saved=0)

Fetching batch 1 (offset=0)...
  -> 50000 records. Total: 50000
Fetching batch 2 (offset=50000)...
  -> 50000 records. Total: 100000
Fetching batch 3 (offset=100000)...
  -> 50000 records. Total: 150000
Fetching batch 4 (offset=150000)...
  -> 50000 records. Total: 200000
Fetching batch 5 (offset=200000)...
  -> 50000 records. Total: 250000
Fetching batch 6 (offset=250000)...
  -> 50000 records. Total: 300000
Fetching batch 7 (offset=300000)...
  -> 50000 records. Total: 350000
Fetching batch 8 (offset=350000)...
  -> 50000 records. Total: 400000
Fetching batch 9 (offset=400000)...
  -> 50000 records. Total: 450000
Fetching batch 10 (offset=450000)...
  -> 50000 records. Total: 500000
Fetching batch 11 (offset=500000)...
  -> 50000 records. Total: 550000
Fetching batch 12 (offset=550000)...
  -> 50000 records. Total: 600000
Fetching batch 13 (offset=600000)...
  -> 50000 records. Total: 650000
Fetching batch 14 (offset=650000)...
  -> 500

In [ ]:
# ── Run download ──────────────────────────────────────────────────────────────
# Set DOWNLOAD_DATA = True if you need to download the data for the first time
# Set CONVERT_PARQUET = True if you already have the CSV but want to convert to parquet
# Set both to False if you already have the parquet and just want to load and explore
DOWNLOAD_DATA    = True
CONVERT_PARQUET  = False

if DOWNLOAD_DATA:
    download_all()
    combine_batches()

if DOWNLOAD_DATA or CONVERT_PARQUET:
    convert_to_parquet()

# ── Load data (uses parquet if available, otherwise CSV) ──────────────────────
# If neither file exists, a FileNotFoundError will tell you to set DOWNLOAD_DATA = True
df = load_data()

In [ ]:
# Load the data
#don't do it this way
#df = pd.read_csv("data/chicago_crime_data/chicago_crimes_2001_2025.csv", low_memory = False)

In [ ]:
# ── Basic info ────────────────────────────────────────────────────────────────
print(df.describe())
print(df.shape)
print(df.dtypes)
print(df.columns.tolist())

In [ ]:
# ── Filter to 2001-2025 ───────────────────────────────────────────────────────
df = df[df['year'].between(2001, 2025)]
print(df['year'].value_counts().sort_index())
print("\nTotal records (2001-2025):", len(df))

In [ ]:
# ── Missing value summary ─────────────────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    "missing_count": missing,
    "missing_pct": missing_pct
}).sort_values("missing_pct", ascending=False)

missing_summary

## Column Alignment Check

All 22 columns are present and correctly ordered — no misalignment detected.

Missing values are due to **genuine data gaps**, not structural issues:

| Column | Missing | Reason |
|--------|---------|--------|
| `ward` / `community_area` | ~614k | Early records do not include these fields |
| `x/y_coordinate` / `latitude` / `longitude` / `location` | ~94k | Addresses that could not be geocoded |
| `location_description` | ~15k | Small number of records with no venue description |
| `district` | 47 | Negligible |

In [ ]:
# ── Data type checks ──────────────────────────────────────────────────────────
# fbi_code and iucr are strings, which is correct - they are category codes
print(df.dtypes)
df.info()



In [ ]:
# Verify arrest and domestic are boolean
print("arrest unique values:", df["arrest"].unique())
print("domestic unique values:", df["domestic"].unique())
#ok good

In [ ]:
# ── Type conversions ──────────────────────────────────────────────────────────

# date and updated_on should be datetime
# errors='coerce' turns unparseable values into NaT rather than crashing
df["date"]       = pd.to_datetime(df["date"], errors="coerce")
df["updated_on"] = pd.to_datetime(df["updated_on"], errors="coerce")

# district, ward, community_area should be integer
# Int64 (capital I) is used because these columns have missing values
# regular int64 cannot store NaN, Int64 can
df["district"]       = pd.to_numeric(df["district"], errors="coerce").astype("Int64")
df["ward"]           = pd.to_numeric(df["ward"], errors="coerce").astype("Int64")
df["community_area"] = pd.to_numeric(df["community_area"], errors="coerce").astype("Int64")

# Recode categorical text columns to save memory
df["primary_type"]        = df["primary_type"].astype("category")
df["description"]         = df["description"].astype("category")
df["location_description"] = df["location_description"].astype("category")

# Verify missing values did not increase after type conversions
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({
    "missing_count": missing,
    "missing_pct": missing_pct
}).sort_values("missing_pct", ascending=False)
missing_summary

In [ ]:
# ── Remove coordinates outside Chicago ───────────────────────────────────────
# Valid Chicago latitude range: 41.6 - 42.1
# Valid Chicago longitude range: -87.9 - -87.5
#df.loc[~df["latitude"].between(41.6, 42.1), "latitude"] = None
#df.loc[~df["longitude"].between(-87.9, -87.5), "longitude"] = None

#print("Latitude range:", df["latitude"].min(), "-", df["latitude"].max())
#print("Longitude range:", df["longitude"].min(), "-", df["longitude"].max())
#commented out incase we don't want ot do it

In [ ]:
# ── Missing data analysis ─────────────────────────────────────────────────────

# Total rows with at least one missing value
print("Rows with at least one missing value:", df.isnull().any(axis=1).sum())
print("That is", round(df.isnull().any(axis=1).sum() / len(df) * 100, 2), "% of the data")

# Which columns are driving the missing values
print("\nMissing values per column for incomplete rows:")
print(df[df.isnull().any(axis=1)][df.columns].isnull().sum())

In [ ]:
# ── Is missing data concentrated in certain years? ────────────────────────────
df["has_missing"] = df.isnull().any(axis=1)

missing_by_year = df.groupby("year")["has_missing"].agg(["sum", "count"])
missing_by_year["pct"] = (missing_by_year["sum"] / missing_by_year["count"] * 100).round(2)
missing_by_year.columns = ["missing_count", "total_rows", "missing_pct"]
print(missing_by_year)
# Note: 2001 has high missing data - worth considering in analysis

# Which specific columns are missing by year
df.groupby("year")[["latitude", "longitude", "ward", "community_area"]].apply(lambda x: x.isnull().sum())

In [ ]:
#double checking for things that may be considered full but just empty text, seems fine
# ── Check for empty strings masquerading as non-null ─────────────────────────
for col in df.select_dtypes(include="object").columns:
    empty_count = (df[col].astype(str).str.strip() == "").sum()
    nan_string  = (df[col].astype(str) == "nan").sum()
    if empty_count > 0 or nan_string > 0:
        print(col, "- empty strings:", empty_count, "| 'nan' strings:", nan_string)
        

## Suggestions for Further Cleaning

The following are suggestions worth discussing with the team before implementing:

- **`location` column** — contains JSON-like strings, redundant with `latitude`/`longitude`, worth removing
- **`updated_on`** — could be recoded as a binary `was_updated` True/False variable since many values are missing
- **`date`** — worth splitting into separate `date` and `time` columns for easier feature engineering
- **Missing data** — ~7% of rows have at least one missing value, concentrated in 2001. Worth discussing whether to drop these rows depending on which features the ML algorithm uses
- **Early years (2001)** — high proportion of missing location data, likely due to geocoding not being available at the time